<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_LeastCost_GenerationPlan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LEAST COST GENERATION PLANNING (Levelized Bus-bar Analysis)

This algorithm:


1.   Compute annual energy E from P and CF
2.   Compute base-year annual cost (fuel, variable O&M, fixed O&M)
3.   Compute a single levelizing factor LF(i,a,n) and multiply those base-year annual cost by LF
4.   Compute annual owning cost as (CAPEX)(FCR). FCR is the fixed-charge rate ans it is assumed to be 20%.
5.   Sum to get total annual levelized cost ($/yr)

6.   Divide by annual MWh to get bus-bar levelized cost ($/MWh)
7.   Rank the 3 units by both metrics

The code can be reused directly (scaled to any number of generators) without modifying the computational core. Information must be manually written in the code (later upgrade would allow to use Excel spreadsheet).

Consider that:


*   We are comparing same capacity, same capacity factor CF, same evaluation horizon
*   Then, ranking is purely driven by (CAPEX)(FCR), (heat rate)(fuel cost), O&M structure
*   Later, we can introduce different forced outage rates (FOR), different capacity factors, different lifetimes, different escalation rates.




In [1]:
"""
Least-Cost Generation Planning (Levelized Bus-Bar Analysis) for 3 or more Generators

This script follows the same procedure shown in the attached example:
- Computes annual energy from capacity and capacity factor.
- Computes base-year annual costs (fuel, variable O&M, fixed O&M).
- Applies a single "levelizing factor" (based on discount rate and escalation) to:
    * fuel cost
    * variable O&M
    * fixed O&M
  (This matches your teaching assumption and the example PDF workflow.)
- Computes annual owning cost using a GIVEN levelized fixed-charge rate (FCR).
- Produces:
    * Total annual levelized cost [$ per year]
    * Bus-bar levelized cost [$ per MWh]
- Ranks alternatives by both metrics.

Author: (you / your course notebook)
"""

from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple


# -----------------------------
# Data structure for one unit
# -----------------------------
@dataclass
class GeneratorUnit:
    """
    Stores input data for a candidate generating unit.
    All costs are in consistent units:
      - Plant cost: $/kW
      - Fixed O&M: $/kW-year
      - Variable O&M: $/MWh
      - Fuel: $/MMBtu
      - Heat rate: Btu/kWh (will be converted to MMBtu/MWh)
    """
    name: str
    capacity_mw: float
    capacity_factor: float
    heat_rate_btu_per_kwh: float
    fuel_cost_per_mmbtu: float
    plant_cost_per_kw: float
    fixed_om_per_kw_year: float
    variable_om_per_mwh: float


# -----------------------------
# Core finance helpers
# -----------------------------
def crf(i: float, n: int) -> float:
    """
    Capital Recovery Factor (CRF):
    Converts a present worth (PV) into an equivalent uniform annual amount (EUAC multiplier).

    CRF(i,n) = i(1+i)^n / ((1+i)^n - 1)
    """
    if i == 0:
        return 1.0 / n
    return (i * (1 + i) ** n) / ((1 + i) ** n - 1)


def levelizing_factor(i: float, a: float, n: int) -> float:
    """
    Levelizing factor used in your example:
    Converts a YEAR-1 escalating annual cost stream into an equivalent UNIFORM annual cost.

    For an escalating series:
      C_t = C_1 * (1+a)^(t-1), for t=1..n
    Present worth:
      PW = C_1 * [1 - ((1+a)/(1+i))^n] / (i - a)     (when i != a)
    Equivalent uniform annual value:
      A = PW * CRF(i,n)
    Thus:
      LF = A / C_1 = CRF(i,n) * [1 - ((1+a)/(1+i))^n] / (i - a)

    NOTE:
    - Requires i != a. If i == a, the closed-form changes; we handle that safely below.
    """
    if n <= 0:
        raise ValueError("n must be positive.")
    if i == a:
        # When i == a, the ratio (1+a)/(1+i) = 1, and the formula divides by zero.
        # In that special case, PW = C1 * n / (1+i) (derived from geometric series limit),
        # so A/C1 = [n/(1+i)] * CRF(i,n).
        return (n / (1 + i)) * crf(i, n)

    ratio = (1 + a) / (1 + i)
    pw_multiplier = (1 - (ratio ** n)) / (i - a)  # PW = C1 * pw_multiplier
    return crf(i, n) * pw_multiplier


# -----------------------------
# Engineering cost helpers
# -----------------------------
def btu_per_kwh_to_mmbtu_per_mwh(heat_rate_btu_per_kwh: float) -> float:
    """
    Convert heat rate from Btu/kWh to MMBtu/MWh.

    Steps:
    - Multiply by 1000 to get Btu/MWh
    - Divide by 1,000,000 to get MMBtu/MWh

    So: (Btu/kWh) * 1000 / 1e6 = (Btu/kWh) / 1000  [MMBtu/MWh]
    Example: 9500 Btu/kWh -> 9.5 MMBtu/MWh
    """
    return heat_rate_btu_per_kwh / 1000.0


def annual_energy_mwh(capacity_mw: float, capacity_factor: float, hours_per_year: float = 8760.0) -> float:
    """
    Annual energy in MWh:
      E = P(MW) * hours/year * CF
    """
    return capacity_mw * hours_per_year * capacity_factor


def evaluate_unit_levelized_busbar(
    unit: GeneratorUnit,
    *,
    n_years: int,
    discount_rate_i: float,
    escalation_rate_a: float,
    fixed_charge_rate_fcr: float,
) -> Dict[str, float]:
    """
    Compute levelized annual owning and operating costs for one unit, using the same approach as the PDF:
    - Compute base-year annual costs for fuel, variable O&M, fixed O&M.
    - Apply the SAME levelizing factor to fuel + variable O&M + fixed O&M.
    - Compute annual owning cost using FCR * initial CAPEX.

    Returns a dict with detailed intermediate values for teaching transparency.
    """
    # 1) Annual energy
    E_mwh = annual_energy_mwh(unit.capacity_mw, unit.capacity_factor)

    # 2) Convert heat rate to MMBtu/MWh
    hr_mmbtu_per_mwh = btu_per_kwh_to_mmbtu_per_mwh(unit.heat_rate_btu_per_kwh)

    # 3) Base-year annual variable costs
    # Fuel cost per MWh = heat rate (MMBtu/MWh) * fuel price ($/MMBtu)
    fuel_cost_per_mwh = hr_mmbtu_per_mwh * unit.fuel_cost_per_mmbtu
    annual_fuel_base = E_mwh * fuel_cost_per_mwh

    # Variable O&M base-year annual
    annual_vom_base = E_mwh * unit.variable_om_per_mwh

    # Fixed O&M base-year annual
    capacity_kw = unit.capacity_mw * 1000.0
    annual_fom_base = capacity_kw * unit.fixed_om_per_kw_year

    # 4) Levelizing factor (same for fuel and O&M per your request)
    LF = levelizing_factor(discount_rate_i, escalation_rate_a, n_years)

    annual_fuel_levelized = annual_fuel_base * LF
    annual_vom_levelized = annual_vom_base * LF
    annual_fom_levelized = annual_fom_base * LF

    # 5) Annual owning cost using fixed-charge rate (given)
    capex_total = capacity_kw * unit.plant_cost_per_kw
    annual_owning = capex_total * fixed_charge_rate_fcr

    # 6) Total annual levelized cost and bus-bar cost
    total_annual_levelized = annual_owning + annual_fuel_levelized + annual_vom_levelized + annual_fom_levelized
    busbar_cost_per_mwh = total_annual_levelized / E_mwh

    return {
        "E_mwh_per_year": E_mwh,
        "heat_rate_mmbtu_per_mwh": hr_mmbtu_per_mwh,
        "fuel_cost_per_mwh": fuel_cost_per_mwh,
        "LF_levelizing_factor": LF,
        "capex_total_$": capex_total,
        "annual_owning_$per_year": annual_owning,
        "annual_fuel_base_$per_year": annual_fuel_base,
        "annual_vom_base_$per_year": annual_vom_base,
        "annual_fom_base_$per_year": annual_fom_base,
        "annual_fuel_levelized_$per_year": annual_fuel_levelized,
        "annual_vom_levelized_$per_year": annual_vom_levelized,
        "annual_fom_levelized_$per_year": annual_fom_levelized,
        "total_annual_levelized_$per_year": total_annual_levelized,
        "busbar_levelized_cost_$per_mwh": busbar_cost_per_mwh,
    }


def rank_units(results: List[Tuple[str, Dict[str, float]]]) -> Dict[str, List[Tuple[str, float]]]:
    """
    Create rankings by:
      - total annual levelized cost ($/yr)
      - bus-bar levelized cost ($/MWh)
    """
    annual_sorted = sorted(
        [(name, r["total_annual_levelized_$per_year"]) for name, r in results],
        key=lambda x: x[1]
    )
    busbar_sorted = sorted(
        [(name, r["busbar_levelized_cost_$per_mwh"]) for name, r in results],
        key=lambda x: x[1]
    )
    return {
        "rank_by_total_annual_levelized_$per_year": annual_sorted,
        "rank_by_busbar_levelized_cost_$per_mwh": busbar_sorted,
    }


def print_summary_table(results: List[Tuple[str, Dict[str, float]]]) -> None:
    """
    Print a compact comparison table (teaching-friendly).
    """
    print("\n=== Levelized Bus-Bar Comparison (3 Units) ===")
    header = (
        f"{'Unit':<10} | {'E (MWh/yr)':>12} | {'Own $/yr':>12} | "
        f"{'Fuel Lvl $/yr':>14} | {'O&M Lvl $/yr':>13} | {'Total Lvl $/yr':>15} | {'$/MWh':>10}"
    )
    print(header)
    print("-" * len(header))

    for name, r in results:
        om_lvl = r["annual_vom_levelized_$per_year"] + r["annual_fom_levelized_$per_year"]
        print(
            f"{name:<10} | "
            f"{r['E_mwh_per_year']:>12,.0f} | "
            f"{r['annual_owning_$per_year']:>12,.0f} | "
            f"{r['annual_fuel_levelized_$per_year']:>14,.0f} | "
            f"{om_lvl:>13,.0f} | "
            f"{r['total_annual_levelized_$per_year']:>15,.0f} | "
            f"{r['busbar_levelized_cost_$per_mwh']:>10.2f}"
        )


def main():
    # -----------------------------
    # Global assumptions (given)
    # -----------------------------
    n_years = 20
    discount_rate_i = 0.10      # present-worth rate
    escalation_rate_a = 0.06    # fuel price escalation (applied also to O&M per your request)
    fixed_charge_rate_fcr = 0.20
    capacity_factor = 0.70
    capacity_mw = 400.0

    # -----------------------------
    # Define the 3 candidate units
    # -----------------------------
    units = [
        GeneratorUnit(
            name="UNIT 1",
            capacity_mw=capacity_mw,
            capacity_factor=capacity_factor,
            heat_rate_btu_per_kwh=9500.0,
            fuel_cost_per_mmbtu=2.0,
            plant_cost_per_kw=1500.0,
            fixed_om_per_kw_year=20.0,
            variable_om_per_mwh=5.0,
        ),
        GeneratorUnit(
            name="UNIT 2",
            capacity_mw=capacity_mw,
            capacity_factor=capacity_factor,
            heat_rate_btu_per_kwh=8500.0,
            fuel_cost_per_mmbtu=2.0,
            plant_cost_per_kw=700.0,
            fixed_om_per_kw_year=9.0,
            variable_om_per_mwh=3.0,
        ),
        GeneratorUnit(
            name="UNIT 3",
            capacity_mw=capacity_mw,
            capacity_factor=capacity_factor,
            heat_rate_btu_per_kwh=11000.0,
            fuel_cost_per_mmbtu=6.0,
            plant_cost_per_kw=350.0,
            fixed_om_per_kw_year=1.0,
            variable_om_per_mwh=5.0,
        ),
    ]

    # Compute LF once and print it (should match ~1.537 for i=10%, a=6%, n=20)
    LF = levelizing_factor(discount_rate_i, escalation_rate_a, n_years)
    print("=== Global Assumptions ===")
    print(f"Evaluation horizon n: {n_years} years")
    print(f"Discount rate i: {discount_rate_i:.2%}/yr")
    print(f"Escalation rate a (applied to fuel and O&M): {escalation_rate_a:.2%}/yr")
    print(f"Fixed-charge rate (given): {fixed_charge_rate_fcr:.2%}/yr")
    print(f"Capacity factor: {capacity_factor:.2%}")
    print(f"Computed levelizing factor LF: {LF:.6f}  (expect ~1.537 for i=10%, a=6%, n=20)")

    # Evaluate each unit
    results: List[Tuple[str, Dict[str, float]]] = []
    for u in units:
        r = evaluate_unit_levelized_busbar(
            u,
            n_years=n_years,
            discount_rate_i=discount_rate_i,
            escalation_rate_a=escalation_rate_a,
            fixed_charge_rate_fcr=fixed_charge_rate_fcr,
        )
        results.append((u.name, r))

    # Print compact comparison table
    print_summary_table(results)

    # Rankings
    rankings = rank_units(results)
    print("\n=== Rankings ===")
    print("1) Least cost by TOTAL annual levelized cost ($/yr):")
    for k, (name, val) in enumerate(rankings["rank_by_total_annual_levelized_$per_year"], start=1):
        print(f"   {k}. {name}: ${val:,.0f}/yr")

    print("\n2) Least cost by BUS-BAR levelized cost ($/MWh):")
    for k, (name, val) in enumerate(rankings["rank_by_busbar_levelized_cost_$per_mwh"], start=1):
        print(f"   {k}. {name}: ${val:,.2f}/MWh")

    # Keep everything accessible for future notebook cells:
    # - units: list of GeneratorUnit
    # - results: list of (name, dict)
    # - rankings: dict of ranked lists
    global_state = {
        "assumptions": {
            "n_years": n_years,
            "discount_rate_i": discount_rate_i,
            "escalation_rate_a": escalation_rate_a,
            "fixed_charge_rate_fcr": fixed_charge_rate_fcr,
            "capacity_factor": capacity_factor,
            "capacity_mw": capacity_mw,
            "LF_levelizing_factor": LF,
        },
        "units": [asdict(u) for u in units],
        "results": {name: r for name, r in results},
        "rankings": rankings,
    }

    # Expose in notebook scope if running interactively
    print("\nSaved objects for future cells: units, results, rankings, global_state")


if __name__ == "__main__":
    main()

=== Global Assumptions ===
Evaluation horizon n: 20 years
Discount rate i: 10.00%/yr
Escalation rate a (applied to fuel and O&M): 6.00%/yr
Fixed-charge rate (given): 20.00%/yr
Capacity factor: 70.00%
Computed levelizing factor LF: 1.536606  (expect ~1.537 for i=10%, a=6%, n=20)

=== Levelized Bus-Bar Comparison (3 Units) ===
Unit       |   E (MWh/yr) |     Own $/yr |  Fuel Lvl $/yr |  O&M Lvl $/yr |  Total Lvl $/yr |      $/MWh
--------------------------------------------------------------------------------------------------------
UNIT 1     |    2,452,800 |  120,000,000 |     71,610,760 |    31,137,785 |     222,748,545 |      90.81
UNIT 2     |    2,452,800 |   56,000,000 |     64,072,785 |    16,838,744 |     136,911,529 |      55.82
UNIT 3     |    2,452,800 |   28,000,000 |    248,753,166 |    19,459,579 |     296,212,746 |     120.77

=== Rankings ===
1) Least cost by TOTAL annual levelized cost ($/yr):
   1. UNIT 2: $136,911,529/yr
   2. UNIT 1: $222,748,545/yr
   3. UNIT 3: $29

Previous bus-bar comparison assumed all units have the same installed capacity and same CF, and ignored outage-driven adequacy. In real least-cost generation planning, we must compare options under a common adequacy target (firm capacity requirement).

Because units have different forced outage rates, we typically need to install more nameplate MW for higher-FOR technologies to deliver the same effective (firm) capacity.

This upgraded example enforces a planning requirement:
P_firm = P_installed (1 - FOR).
Then each unit's:


*   CAPEX and fixed O&M scale with required installed MW
*   Annual energy scales with installed MW and its capacity factor
*   Levelizing uses each unit's lifetime, discount rate, escalation rate
*   Comparison is more realistic for planning because it couples adequacy with economics.



In [5]:
# =========================
# CELL 2: Read units from Excel + adequacy via FOR -> effective capacity -> required installed MW
# Upgrades:
#   - Different FOR, CF, lifetime, discount rate, escalation rate, FCR per unit
#   - Required installed MW computed to meet a common firm-capacity target
#   - Same levelized bus-bar workflow, now scaled by adequacy requirement
# =========================

import pandas as pd

# -------------------------
# USER SETTINGS (edit freely)
# -------------------------
excel_path = "/content/GenDataForGenExpansionPlan.xlsx"   # <-- your Excel file name/path
sheet_name = "Units"

# Planning adequacy requirement (common target firm capacity)
target_firm_capacity_mw = 400.0   # MW of effective/firm capacity you want each candidate to satisfy

# Optional: if Excel has blanks for some fields, provide defaults here
DEFAULTS = {
    "DiscountRate": 0.10,
    "EscalationRate": 0.06,
    "FixedChargeRate": 0.20,
    "LifetimeYears": 20,
    "CapacityFactor": 0.70,
    "ForcedOutageRate": 0.07
}

# -------------------------
# 1) Load and validate input table
# -------------------------
df_units = pd.read_excel(excel_path, sheet_name=sheet_name)

required_cols = [
    "Unit", "ForcedOutageRate", "CapacityFactor", "LifetimeYears",
    "DiscountRate", "EscalationRate", "FixedChargeRate",
    "HeatRate_BtuPerkWh", "FuelCost_$/MMBtu",
    "PlantCost_$/kW", "FixedOM_$/kWyr", "VarOM_$/MWh"
]
missing = [c for c in required_cols if c not in df_units.columns]
if missing:
    raise ValueError(f"Excel sheet '{sheet_name}' is missing required columns: {missing}")

# Fill optional blanks with defaults (if any)
for col, default_val in DEFAULTS.items():
    if col in df_units.columns:
        df_units[col] = df_units[col].fillna(default_val)

# Basic numeric validation / type coercion
numeric_cols = [c for c in required_cols if c != "Unit"]
df_units[numeric_cols] = df_units[numeric_cols].apply(pd.to_numeric, errors="raise")

# Sanity checks
if (df_units["ForcedOutageRate"] < 0).any() or (df_units["ForcedOutageRate"] >= 1).any():
    raise ValueError("ForcedOutageRate must be in [0, 1).")

if (df_units["CapacityFactor"] <= 0).any() or (df_units["CapacityFactor"] > 1).any():
    raise ValueError("CapacityFactor must be in (0, 1].")

if (df_units["LifetimeYears"] <= 0).any():
    raise ValueError("LifetimeYears must be positive.")

# -------------------------
# 2) Adequacy scaling: FOR -> effective capacity -> required installed MW
# -------------------------
# Effective capacity fraction (simple planning approximation)
df_units["EffectiveCapacityFraction"] = 1.0 - df_units["ForcedOutageRate"]

# Required installed capacity to meet the target firm capacity
df_units["RequiredInstalledMW"] = target_firm_capacity_mw / df_units["EffectiveCapacityFraction"]

# Annual MWh for each candidate after adequacy scaling
df_units["AnnualMWh"] = df_units["RequiredInstalledMW"] * 8760.0 * df_units["CapacityFactor"]

# -------------------------
# 3) Evaluate each unit using the same levelized bus-bar method
#    but with:
#      capacity_mw = RequiredInstalledMW
#      and per-unit i, a, n, FCR
# -------------------------
results_xl = []  # list of (UnitName, result_dict)

for _, row in df_units.iterrows():
    unit_obj = GeneratorUnit(
        name=str(row["Unit"]),
        capacity_mw=float(row["RequiredInstalledMW"]),                  # scaled to meet firm target
        capacity_factor=float(row["CapacityFactor"]),
        heat_rate_btu_per_kwh=float(row["HeatRate_BtuPerkWh"]),
        fuel_cost_per_mmbtu=float(row["FuelCost_$/MMBtu"]),
        plant_cost_per_kw=float(row["PlantCost_$/kW"]),
        fixed_om_per_kw_year=float(row["FixedOM_$/kWyr"]),
        variable_om_per_mwh=float(row["VarOM_$/MWh"])
    )

    r = evaluate_unit_levelized_busbar(
        unit_obj,
        n_years=int(row["LifetimeYears"]),
        discount_rate_i=float(row["DiscountRate"]),
        escalation_rate_a=float(row["EscalationRate"]),
        fixed_charge_rate_fcr=float(row["FixedChargeRate"]),
    )

    # Add adequacy-relevant fields for reporting
    r["ForcedOutageRate"] = float(row["ForcedOutageRate"])
    r["EffectiveCapacityFraction"] = float(row["EffectiveCapacityFraction"])
    r["RequiredInstalledMW"] = float(row["RequiredInstalledMW"])
    r["TargetFirmCapacityMW"] = float(target_firm_capacity_mw)
    r["LifetimeYears"] = int(row["LifetimeYears"])
    r["DiscountRate"] = float(row["DiscountRate"])
    r["EscalationRate"] = float(row["EscalationRate"])
    r["FixedChargeRate"] = float(row["FixedChargeRate"])

    results_xl.append((unit_obj.name, r))

# -------------------------
# 4) Print comparison + rankings (same two criteria as requested)
# -------------------------
print("\n=== Adequacy-Adjusted Levelized Bus-Bar Analysis ===")
print(f"Target firm capacity requirement: {target_firm_capacity_mw:,.1f} MW")
print("Interpretation: each candidate is scaled so InstalledMW*(1-FOR) = target firm MW.\n")

# Compact table (reuses your original function; it will reflect scaled MW via higher/lower annual costs)
print_summary_table(results_xl)

rankings_xl = rank_units(results_xl)

print("\n=== Rankings (Adequacy-Adjusted) ===")
print("1) Least cost by TOTAL annual levelized cost ($/yr):")
for k, (name, val) in enumerate(rankings_xl["rank_by_total_annual_levelized_$per_year"], start=1):
    print(f"   {k}. {name}: ${val:,.0f}/yr")

print("\n2) Least cost by BUS-BAR levelized cost ($/MWh):")
for k, (name, val) in enumerate(rankings_xl["rank_by_busbar_levelized_cost_$per_mwh"], start=1):
    print(f"   {k}. {name}: ${val:,.2f}/MWh")

# -------------------------
# 5) Save a "state" dictionary for future cells (e.g., Monte Carlo fuel risk later)
# -------------------------
global_state_xl = {
    "excel_path": excel_path,
    "sheet_name": sheet_name,
    "target_firm_capacity_mw": target_firm_capacity_mw,
    "df_units": df_units,                       # pandas table with computed installed MW, annual MWh, etc.
    "results": {name: r for name, r in results_xl},
    "rankings": rankings_xl
}

print("\nSaved for future cells: df_units, results_xl, rankings_xl, global_state_xl")


=== Adequacy-Adjusted Levelized Bus-Bar Analysis ===
Target firm capacity requirement: 400.0 MW
Interpretation: each candidate is scaled so InstalledMW*(1-FOR) = target firm MW.


=== Levelized Bus-Bar Comparison (3 Units) ===
Unit       |   E (MWh/yr) |     Own $/yr |  Fuel Lvl $/yr |  O&M Lvl $/yr |  Total Lvl $/yr |      $/MWh
--------------------------------------------------------------------------------------------------------
UNIT 1     |    2,637,419 |  129,032,258 |     77,000,817 |    33,481,490 |     239,514,565 |      90.81
UNIT 2     |    2,950,737 |   58,947,368 |     83,668,270 |    20,081,541 |     162,697,179 |      55.14
UNIT 3     |    1,904,348 |   30,434,783 |    199,645,230 |    15,815,261 |     245,895,274 |     129.12

=== Rankings (Adequacy-Adjusted) ===
1) Least cost by TOTAL annual levelized cost ($/yr):
   1. UNIT 2: $162,697,179/yr
   2. UNIT 1 : $239,514,565/yr
   3. UNIT 3: $245,895,274/yr

2) Least cost by BUS-BAR levelized cost ($/MWh):
   1. UNIT 2: $